# Latent Failure Forecasting PoC on Google Colab

Probes whether a trajectory's eventual success/failure becomes more decodable from the residual stream as the agent nears its own conclusion. Collects trajectories on GPU, then trains logistic-regression probes (layer x relative-position).

1. `Runtime > Change runtime type > T4 GPU > Save`
2. **`Runtime > Restart session`** (CUDA is not visible until you restart after enabling GPU)
3. Run cells **in order**, one at a time. Do not start another cell while one is running.
4. **After the install cell, `Runtime > Restart session` again, then continue from the GPU-check cell.** The install removes a mismatched `torchvision` (it has nothing to do with this PoC) that otherwise makes `transformers` fail to import with `operator torchvision::nms does not exist`.

**About `^C` right after `Loading weights`:** that is the runtime **killing the process when it runs out of RAM** during model loading — not you. The model now loads in fp16 to avoid it, and the pipeline runs in-process so any real error is shown.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'CUDA not visible. Set runtime to T4 GPU, then Runtime > Restart session, then re-run.'
)
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
import psutil
print(f'System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')

In [ ]:
import os
if os.path.basename(os.getcwd()) != 'algoverse-PoC':
    if not os.path.isdir('algoverse-PoC'):
        !git clone https://github.com/abdelmagid07/algoverse-PoC.git
    %cd algoverse-PoC
!git pull --ff-only

In [ ]:
!pip install -q -r requirements.txt
# torchvision is unused here. pip can leave it mismatched with the torch that
# transformer_lens pulls in, which makes `transformers` fail on import with
# "operator torchvision::nms does not exist". Removing it makes transformers
# skip torchvision entirely. After this cell, do Runtime > Restart session,
# then continue from the GPU-check cell (a half-imported torchvision lingers
# in the crashed kernel until you restart).
!pip uninstall -q -y torchvision || true
print("\nDeps installed. Now: Runtime > Restart session, then run from the GPU-check cell.")

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token: '))

In [ ]:
# Smoke test: load the model in-process (fp16). Confirms GPU + that loading does not OOM.
import os
os.environ['VERITAS_DEVICE'] = 'cuda'

from interp.activation_cache import load_model, log_device_choice
log_device_choice()
model = load_model()
print('Loaded:', model.cfg.model_name, '| dtype:', next(model.parameters()).dtype,
      '| device:', next(model.parameters()).device)
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# Full pipeline, in-process (~15-30 min). Wait for '=== Done ==='.
# If the runtime crashes/disconnects, just re-run this cell - it resumes from checkpoints.
import run_pipeline
run_pipeline.main()

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

for p in ['results/probe_results.json', 'results/accuracy_by_position.png', 'results/poc_summary.md']:
    assert Path(p).exists(), f'Missing {p} - run the pipeline cell to completion'

for img in ['results/accuracy_by_position.png', 'results/accuracy_by_layer.png', 'results/probe_heatmap.png']:
    display(Image(img))
display(Markdown(open('results/poc_summary.md').read()))